# Laboratório — Mini-batch, epoch e embaralhamento

Este laboratório isola o **estimador de gradiente**: todos os gradientes comparados são avaliados nos mesmos parâmetros. Não há atualização por SGD, que pertence à Aula 18.

**Dependências:** Python ≥ 3.11, NumPy ≥ 1.26 e Matplotlib ≥ 3.8.  
**Seed-base:** `20260917`.  
**Dados:** regressão linear sintética, documentada e sem download externo.


## 1. Ambiente reproduzível

Além da seed, registramos versões e o bit generator. O NumPy não garante o mesmo bitstream entre versões futuras.


In [ ]:
import platform
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("error")
BASE_SEED = 20260917
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Bit generator: {np.random.default_rng(BASE_SEED).bit_generator.__class__.__name__}")


## 2. Dados e unidade de análise

Cada linha é um exemplo independente. Criamos 103 observações para que o último lote de tamanho nominal 16 seja parcial.


In [ ]:
rng_data = np.random.default_rng(BASE_SEED)
N, D, BATCH_SIZE = 103, 4, 16
X = rng_data.normal(size=(N, D))
w_true = np.array([2.0, -1.0, 0.5, 1.2])
y = X @ w_true + rng_data.normal(scale=0.30, size=N)
ids = np.arange(N)

assert X.shape == (103, 4)
assert y.shape == ids.shape == (103,)
assert np.isfinite(X).all() and np.isfinite(y).all()
print(f"X={X.shape}; y={y.shape}; IDs únicos={len(np.unique(ids))}")


## 3. Iterador por índices

A ordem de cada época é derivada de `(seed, epoch)`. Uma única sequência de índices governa entradas, alvos e metadados.


In [ ]:
def batch_indices(n, batch_size, *, seed, epoch, shuffle=True, drop_last=False):
    if n <= 0 or batch_size <= 0:
        raise ValueError("n e batch_size devem ser positivos")
    indices = np.arange(n)
    if shuffle:
        seed_sequence = np.random.SeedSequence([seed, epoch])
        indices = np.random.default_rng(seed_sequence).permutation(indices)
    stop = n if not drop_last else n - (n % batch_size)
    for start in range(0, stop, batch_size):
        batch = indices[start:min(start + batch_size, n)]
        if len(batch) == batch_size or not drop_last:
            yield batch

normal = list(batch_indices(N, BATCH_SIZE, seed=BASE_SEED, epoch=0))
dropped = list(batch_indices(N, BATCH_SIZE, seed=BASE_SEED, epoch=0, drop_last=True))
assert [len(b) for b in normal] == [16, 16, 16, 16, 16, 16, 7]
assert [len(b) for b in dropped] == [16] * 6
print("Tamanhos sem drop_last:", [len(b) for b in normal])
print("Cobertura com drop_last:", sum(map(len, dropped)), "de", N)


## 4. Cobertura: nem repetição nem omissão acidental

Uma época sem `drop_last` deve ser uma permutação exata de `0..N-1`.


In [ ]:
flat = np.concatenate(normal)
assert len(flat) == N
assert len(np.unique(flat)) == N
assert np.array_equal(np.sort(flat), np.arange(N))

flat_drop = np.concatenate(dropped)
missing_drop = np.setdiff1d(np.arange(N), flat_drop)
assert len(missing_drop) == N % BATCH_SIZE == 7
print(f"Cobertura completa: {len(flat)}/{N}; duplicações: {len(flat)-len(np.unique(flat))}")
print("IDs omitidos por drop_last nesta época:", missing_drop.tolist())


## 5. Alinhamento entre `X`, `y` e IDs

O teste abaixo constrói um alvo determinístico por ID. Se qualquer array usar outra permutação, o contrato falha.


In [ ]:
X_contract = np.column_stack([ids, ids**2])
y_contract = 3 * ids - 7
for idx in normal:
    assert np.array_equal(X_contract[idx, 0], ids[idx])
    assert np.array_equal(y_contract[idx], 3 * X_contract[idx, 0] - 7)

rng_bad = np.random.default_rng(BASE_SEED)
p_x = rng_bad.permutation(N)
p_y = rng_bad.permutation(N)
bad_alignment_rate = np.mean(y_contract[p_y] == 3 * X_contract[p_x, 0] - 7)
assert bad_alignment_rate < 0.10
print(f"Alinhamento com índice único: 100%; com permutações independentes: {bad_alignment_rate:.3%}")


## 6. Retomada direta de uma época

A época 3 pode ser reconstruída sem executar as épocas 0, 1 e 2.


In [ ]:
def epoch_order(epoch):
    return np.concatenate(list(batch_indices(N, BATCH_SIZE, seed=BASE_SEED, epoch=epoch)))

order_0a = epoch_order(0)
order_0b = epoch_order(0)
order_1 = epoch_order(1)
order_3_direct = epoch_order(3)
orders_sequential = [epoch_order(e) for e in range(4)]

assert np.array_equal(order_0a, order_0b)
assert not np.array_equal(order_0a, order_1)
assert np.array_equal(order_3_direct, orders_sequential[3])
print("Época 0 repetível:", np.array_equal(order_0a, order_0b))
print("Épocas 0 e 1 distintas:", not np.array_equal(order_0a, order_1))
print("Retomada direta da época 3:", np.array_equal(order_3_direct, orders_sequential[3]))


## 7. Gradiente completo em parâmetros fixos

Para $\ell_i(w)=\tfrac12(x_i^\top w-y_i)^2$, o gradiente individual é $(x_i^\top w-y_i)x_i$.


In [ ]:
def per_example_gradients(Xb, yb, w):
    residual = Xb @ w - yb
    return residual[:, None] * Xb

def mean_gradient(Xb, yb, w):
    return per_example_gradients(Xb, yb, w).mean(axis=0)

w0 = np.array([0.25, -0.10, 0.15, 0.05])
population_grads = per_example_gradients(X, y, w0)
full_grad = population_grads.mean(axis=0)
assert full_grad.shape == (D,) and np.isfinite(full_grad).all()
print("Gradiente completo:", np.round(full_grad, 6))


## 8. Agregação ponderada do último lote

Com parâmetros fixos, a média ponderada dos batches recupera exatamente o gradiente completo. A média simples das médias dá ao lote de 7 o mesmo peso de um lote de 16.


In [ ]:
ordered_batches = list(batch_indices(N, BATCH_SIZE, seed=BASE_SEED, epoch=0, shuffle=False))
batch_grads = np.array([mean_gradient(X[idx], y[idx], w0) for idx in ordered_batches])
sizes = np.array([len(idx) for idx in ordered_batches])
weighted_grad = np.average(batch_grads, axis=0, weights=sizes)
naive_grad = batch_grads.mean(axis=0)
weighted_error = np.max(np.abs(weighted_grad - full_grad))
naive_error = np.linalg.norm(naive_grad - full_grad)

assert weighted_error < 1e-14
assert naive_error > 1e-3
print(f"Erro máximo ponderado: {weighted_error:.3e}")
print(f"Erro L2 da média ingênua: {naive_error:.6f}")


## 9. Redução `mean` versus `sum`

Num lote completo de 16, somar gradientes multiplica sua escala por 16. A política de redução faz parte do contrato com o learning rate.


In [ ]:
idx16 = ordered_batches[0]
g_mean = population_grads[idx16].mean(axis=0)
g_sum = population_grads[idx16].sum(axis=0)
ratio = np.linalg.norm(g_sum) / np.linalg.norm(g_mean)
assert np.allclose(g_sum, len(idx16) * g_mean)
assert np.isclose(ratio, 16.0)
print(f"Razão de normas sum/mean: {ratio:.6f}")


## 10. Amostragem com reposição: média e variância

Para cada batch size, sorteamos 4.000 lotes independentes, sempre no mesmo $w_0$. A estatística é o erro quadrático médio vetorial em relação ao gradiente completo.


In [ ]:
rng_mc = np.random.default_rng(BASE_SEED + 1)
batch_sizes = np.array([1, 4, 16, 64])
trials = 4000
population_trace = np.mean(np.sum((population_grads - full_grad) ** 2, axis=1))
observed_wr, theoretical_wr, mean_errors = [], [], []

for b in batch_sizes:
    draw = rng_mc.integers(0, N, size=(trials, b))
    estimates = population_grads[draw].mean(axis=1)
    observed_wr.append(np.mean(np.sum((estimates - full_grad) ** 2, axis=1)))
    theoretical_wr.append(population_trace / b)
    mean_errors.append(np.linalg.norm(estimates.mean(axis=0) - full_grad))

observed_wr = np.array(observed_wr)
theoretical_wr = np.array(theoretical_wr)
mean_errors = np.array(mean_errors)
relative = observed_wr / theoretical_wr
assert np.all((relative > 0.90) & (relative < 1.10))
assert np.max(mean_errors) < 0.12

for b, obs, theo, err in zip(batch_sizes, observed_wr, theoretical_wr, mean_errors):
    print(f"B={b:2d}: variância observada={obs:.6f}; teoria={theo:.6f}; erro da média={err:.6f}")


### Gráfico: ruído do estimador por batch size

O gráfico a seguir usa eixos logarítmicos. **Texto alternativo:** duas curvas descendentes, observada e teórica, quase sobrepostas; quadruplicar o lote reduz aproximadamente quatro vezes o erro quadrático médio do gradiente.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(batch_sizes, observed_wr, "o-", label="observado")
ax.loglog(batch_sizes, theoretical_wr, "--", label="teoria 1/B")
ax.set(xlabel="batch size B", ylabel="E[||ĝ - g||²]", title="Variância do gradiente médio")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)


## 11. Sem reposição: correção de população finita

Repetimos o experimento usando amostras uniformes sem reposição. Quando $B=N$, cada lote é o conjunto completo e a variância zera.


In [ ]:
rng_wor = np.random.default_rng(BASE_SEED + 2)
sizes_wor = np.array([1, 16, 64, N])
observed_wor, theoretical_wor = [], []

for b in sizes_wor:
    if b == N:
        estimates = np.repeat(full_grad[None, :], trials, axis=0)
    else:
        estimates = np.array([
            population_grads[rng_wor.choice(N, size=b, replace=False)].mean(axis=0)
            for _ in range(trials)
        ])
    observed_wor.append(np.mean(np.sum((estimates - full_grad) ** 2, axis=1)))
    theoretical_wor.append(((N - b) / (N - 1)) * population_trace / b)

observed_wor = np.array(observed_wor)
theoretical_wor = np.array(theoretical_wor)
assert np.allclose(observed_wor[-1], 0.0, atol=1e-28)
assert np.all(np.abs(observed_wor[:-1] / theoretical_wor[:-1] - 1) < 0.10)
for b, obs, theo in zip(sizes_wor, observed_wor, theoretical_wor):
    print(f"B={b:3d}: observada={obs:.8f}; correção finita={theo:.8f}")


## 12. Ordem acidental e primeiro lote

Ordenamos os exemplos por alvo. O primeiro lote fixo deixa de representar o conjunto. A média do primeiro lote sobre muitas permutações aproxima o gradiente completo.


In [ ]:
sort_idx = np.argsort(y)
X_sorted, y_sorted = X[sort_idx], y[sort_idx]
fixed_first = mean_gradient(X_sorted[:BATCH_SIZE], y_sorted[:BATCH_SIZE], w0)
fixed_first_error = np.linalg.norm(fixed_first - full_grad)

first_gradients = []
for epoch in range(1000):
    first = next(batch_indices(N, BATCH_SIZE, seed=BASE_SEED, epoch=epoch))
    first_gradients.append(mean_gradient(X[first], y[first], w0))
first_gradients = np.array(first_gradients)
shuffled_mean_error = np.linalg.norm(first_gradients.mean(axis=0) - full_grad)

assert fixed_first_error > 2 * shuffled_mean_error
assert shuffled_mean_error < 0.08
print(f"Erro do primeiro lote em dados ordenados: {fixed_first_error:.6f}")
print(f"Erro da média de 1.000 primeiros lotes embaralhados: {shuffled_mean_error:.6f}")


## 13. `drop_last` pode excluir um subgrupo inteiro

Marcamos os sete últimos exemplos como raros. Sem shuffle, eles formam exatamente a cauda descartada. Com shuffle, a exclusão média gira entre os exemplos, mas a época continua incompleta.


In [ ]:
rare = np.zeros(N, dtype=bool)
rare[-7:] = True
fixed_kept = np.concatenate(list(batch_indices(N, BATCH_SIZE, seed=BASE_SEED, epoch=0, shuffle=False, drop_last=True)))
fixed_rare_omitted = rare.sum() - rare[fixed_kept].sum()

omitted_rare = []
for epoch in range(500):
    kept = np.concatenate(list(batch_indices(N, BATCH_SIZE, seed=BASE_SEED, epoch=epoch, drop_last=True)))
    omitted_rare.append(rare.sum() - rare[kept].sum())
mean_omitted = float(np.mean(omitted_rare))
expected = 7 * 7 / N

assert fixed_rare_omitted == 7
assert abs(mean_omitted - expected) < 0.10
print(f"Raros omitidos com ordem fixa: {fixed_rare_omitted}/7")
print(f"Média com shuffle: {mean_omitted:.3f}; esperança: {expected:.3f}")


## 14. Shuffle não corrige leakage de entidade nem de tempo

Uma divisão aleatória por linha sobrepõe pacientes. Uma divisão por entidade zera a sobreposição. Em dados temporais, a divisão causal satisfaz `max(treino) < min(validação)`.


In [ ]:
entity_ids = np.repeat(np.arange(30), 4)
rng_split = np.random.default_rng(BASE_SEED + 3)
row_order = rng_split.permutation(len(entity_ids))
row_train, row_val = row_order[:80], row_order[80:]
row_overlap = np.intersect1d(entity_ids[row_train], entity_ids[row_val])

entity_order = rng_split.permutation(30)
train_entities, val_entities = entity_order[:20], entity_order[20:]
group_overlap = np.intersect1d(train_entities, val_entities)

times = np.arange(120)
time_train, time_val = times[:80], times[80:]
random_train, random_val = row_order[:80], row_order[80:]

assert len(row_overlap) > 0
assert len(group_overlap) == 0
assert time_train.max() < time_val.min()
assert not (random_train.max() < random_val.min())
print(f"Entidades sobrepostas no split por linha: {len(row_overlap)}")
print(f"Entidades sobrepostas no split por grupo: {len(group_overlap)}")
print(f"Split temporal causal: {time_train.max()} < {time_val.min()}")


## 15. Auditoria final

Os contratos cobrem shapes, cobertura, alinhamento, reprodutibilidade, redução, teoria amostral, ordem, subgrupos e splits.


In [ ]:
checks = {
    "shape_X": X.shape == (N, D),
    "shape_y": y.shape == (N,),
    "ids_unicos": len(np.unique(ids)) == N,
    "sete_batches": len(normal) == 7,
    "ultimo_com_sete": len(normal[-1]) == 7,
    "cobertura_total": len(flat) == N,
    "sem_duplicacao": len(np.unique(flat)) == N,
    "drop_last_96": len(flat_drop) == 96,
    "alinhamento": bad_alignment_rate < 0.10,
    "epoca_repetivel": np.array_equal(order_0a, order_0b),
    "epocas_distintas": not np.array_equal(order_0a, order_1),
    "retomada": np.array_equal(order_3_direct, orders_sequential[3]),
    "gradiente_finito": np.isfinite(full_grad).all(),
    "ponderacao_exata": weighted_error < 1e-14,
    "media_ingenua_falha": naive_error > 1e-3,
    "sum_mean": np.isclose(ratio, 16.0),
    "reposicao_teoria": np.all((relative > 0.90) & (relative < 1.10)),
    "sem_reposicao_teoria": np.all(np.abs(observed_wor[:-1] / theoretical_wor[:-1] - 1) < 0.10),
    "full_batch_sem_variancia": np.isclose(observed_wor[-1], 0.0, atol=1e-28),
    "shuffle_desenviesa_primeiro": shuffled_mean_error < 0.08,
    "drop_last_detectado": fixed_rare_omitted == 7,
    "leakage_por_linha": len(row_overlap) > 0,
    "split_por_grupo": len(group_overlap) == 0,
    "tempo_causal": time_train.max() < time_val.min(),
}
failed = [name for name, ok in checks.items() if not ok]
assert not failed, failed
print(f"Auditoria: {sum(checks.values())}/{len(checks)} contratos aprovados.")


## Conclusão

O laboratório confirmou que:

- uma época sem descarte cobre 103 IDs uma vez; com `drop_last`, cobre 96;
- a agregação ponderada recupera o gradiente completo em parâmetros fixos;
- a variância acompanha $1/B$ com reposição e a correção finita sem reposição;
- uma permutação única preserva pares, enquanto permutações independentes os quebram;
- shuffle reduz viés de ordem, mas não corrige leakage por entidade ou tempo.

Na Aula 18, o iterador será acoplado à atualização $\theta \leftarrow \theta-\eta\widehat g$.
